In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install torch-fidelity

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.6/85.6 kB 4.9 MB/s eta 0:00:00


In [3]:
import os
from PIL import Image
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
import torch
import torch.optim as optim
import matplotlib.pyplot as plt
import time
import numpy as np
import math
import sys
from torch_fidelity import calculate_metrics

In [4]:
REPO_DIR = '/content/Texture_synthesis'
os.chdir('/content')

if not os.path.exists(REPO_DIR):
  !git clone https://github.com/ValentinaEmili/Texture-synthesis.git Texture_synthesis

if REPO_DIR not in sys.path:
  sys.path.append(REPO_DIR)

Cloning into 'Texture_synthesis'...
remote: Enumerating objects: 855, done.
remote: Counting objects: 100% (204/204), done.
remote: Compressing objects: 100% (178/178), done.
remote: Total 855 (delta 98), reused 9 (delta 9), pack-reused 651 (from 2)
Receiving objects: 100% (855/855), 101.92 MiB | 16.72 MiB/s, done.
Resolving deltas: 100% (385/385), done.


In [5]:
!pip install import-ipynb -q
import import_ipynb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 74.2 MB/s eta 0:00:00


In [6]:
model_type = 'vq_vae'
#model_type = 'vq_gan'
#model_type = 'vq_gan_tt'

split = '1'
#split = '2'
#split = '1_2'

train_file = 'train_' + split + '.txt'
val_file = 'val_' + split + '.txt'
test_file = 'test_' + split + '.txt'

In [22]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 8

from Texture_synthesis.generation.Transformers.Transformers import Transformer

if model_type == 'vq_vae':
  from Texture_synthesis.codebook.VQ_VAE import VQ_VAE
  generator = VQ_VAE().to(device)
  seq_len = 1024
  num_embeddings = size = 512
  quant_save_path = f'/content/drive/MyDrive/DeepLearning/dtd/checkpoints/{model_type}/best_{split}.pth'

elif model_type == 'vq_gan':
  from Texture_synthesis.codebook.VQGAN import VQGAN, Discriminator
  generator = VQGAN().to(device)
  seq_len = 1024
  num_embeddings = size = 512
  quant_save_path = f'/content/drive/MyDrive/DeepLearning/dtd/checkpoints/{model_type}/gen_best_{split}.pth'

else:
  from Texture_synthesis.codebook.VQGAN_TT import VQGAN, Discriminator
  generator = VQGAN().to(device)
  seq_len = 256
  num_embeddings = 1024
  size = 256
  quant_save_path = f'/content/drive/MyDrive/DeepLearning/dtd/checkpoints/vq_gan/taming_transformers/gen_best_{split}.pth'

checkpoint = torch.load(quant_save_path, weights_only=True)
generator.load_state_dict(checkpoint['model_state_dict'])

best_save_path = f'/content/drive/MyDrive/DeepLearning/dtd/checkpoints/transformers/split_{split}/{model_type}_best.pth'
transformer = Transformer(num_embeddings=num_embeddings, seq_len=seq_len, n_layers=8).to(device)
optimizer = optim.Adam(transformer.parameters(), lr=2e-4, betas=(0.9, 0.999))

out_dir = f'/content/drive/MyDrive/DeepLearning/dtd/generated_images/split_{split}/{model_type}'
real_dir = f"drive/MyDrive/DeepLearning/dtd/test_images/split_{split}"
recon_save_dir = f'/content/drive/MyDrive/DeepLearning/dtd/transformer_reconstructed_images/split_{split}/{model_type}'

In [10]:
train_transform = transforms.Compose([
        transforms.Resize(size),
        transforms.RandomCrop(size),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])

eval_transform = transforms.Compose([
        transforms.Resize(size),
        transforms.CenterCrop(size),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])

class DTD_Dataset(Dataset):
    def __init__(self, root, file_list, transform=None, class_to_idx=None):
        self.root = root
        self.transform = transform

        with open(file_list, mode='r', encoding='utf-8') as f:
            self.files = [line.strip() for line in f if line.strip()]

        if class_to_idx is None:
          unique_classes = sorted({os.path.normpath(p).split(os.sep)[0] for p in self.files})
          self.class_to_idx = {class_name: i for i, class_name in enumerate(unique_classes)}
        else:
          self.class_to_idx = class_to_idx

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        relative_path = self.files[idx]
        image_path = os.path.join(self.root, relative_path)
        img = Image.open(image_path).convert('RGB')
        rel_norm = os.path.normpath(relative_path)
        string_label = rel_norm.split(os.sep, 1)[0]
        label = self.class_to_idx[string_label]

        if self.transform:
            img = self.transform(img)

        return img, label

In [11]:
path_images = "drive/MyDrive/DeepLearning/dtd/images"
path_labels = "drive/MyDrive/DeepLearning/dtd/labels"
train_dataset = DTD_Dataset(path_images, os.path.join(path_labels, train_file), train_transform)
class_to_idx = train_dataset.class_to_idx
val_dataset = DTD_Dataset(path_images, os.path.join(path_labels, val_file), eval_transform, class_to_idx=class_to_idx)
test_dataset = DTD_Dataset(path_images, os.path.join(path_labels, test_file), eval_transform, class_to_idx=class_to_idx)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

In [12]:
@torch.no_grad()
def extract_codebook_indices(quantizer, data):
  quantizer.eval()

  embedding_dim = quantizer.vq.embedding_dim
  embeddings = quantizer.vq.embeddings.weight                               # (num_embeddings, embed_dim)

  z = quantizer.encoder(data)                                               # (batch_size, embed_dim, h, w)
  z_flattened = z.permute(0, 2, 3, 1).contiguous().view(-1, embedding_dim)  # (batch_size * h * w, embed_dim)

  distances = (torch.sum(z_flattened**2, dim=1, keepdim=True)
                    + torch.sum(embeddings**2, dim=1)
                    - 2 * torch.matmul(z_flattened, embeddings.t()))        # (batch_size * h * w, num_embeddings)

  indices = torch.argmin(distances, dim=1)        # (batch_size * h * w)
  return indices.reshape(z.shape[0],-1)           # (batch_size, seq_len)

# Validation

In [13]:
@torch.no_grad()
def validate(val_loader, transformer, quantizer, device):
  transformer.eval()
  quantizer.eval()

  transformer.to(device)
  quantizer.to(device)

  bos_token_id = transformer.num_embeddings

  total_loss, total_accuracy, total_tokens = 0.0, 0.0, 0.0

  for batch_idx, (data, _) in enumerate(val_loader):
    data = data.to(device)

    indices = extract_codebook_indices(quantizer, data)
    b, t = indices.shape                      # (batch_size, seq_len)

    bos_tokens = torch.full((b, 1), bos_token_id, dtype=torch.long, device=device)
    full_sequence = torch.cat([bos_tokens, indices], dim=1) # (batch_size, seq_len)

    inputs = full_sequence[:, :-1]            # [BOS, c1, c2, ..., c_{t-1}]
    targets = full_sequence[:, 1:]            # [c1, c2, ..., c_{t-1}, c_{t}]

    num_tokens = targets.numel()
    total_tokens += num_tokens

    logits = transformer(inputs)              # (batch_size, seq_len, num_embeddings)

    predictions = torch.argmax(logits, dim=-1)
    accuracy = (predictions == targets).sum().item()

    targets = targets.reshape(-1)                             # (batch_size * seq_len)
    logits = logits.reshape(-1, transformer.num_embeddings) # (batch_size * seq_len, num_embeddings)

    loss = F.cross_entropy(logits, targets, reduction="sum")

    total_loss += loss.item()
    total_accuracy += accuracy

  mean_loss = total_loss / total_tokens
  mean_accuracy = (total_accuracy / total_tokens) * 100
  return mean_loss, mean_accuracy

# Training

In [19]:
def train_and_validation(train_loader, val_loader, transformer, quantizer, optimizer, device, best_val_loss=float('inf'), epochs=50):
  transformer.to(device)
  quantizer.to(device)

  bos_token_id = transformer.num_embeddings

  for epoch in range(epochs):
    transformer.train()
    quantizer.eval()
    total_loss, total_accuracy, total_tokens = 0.0, 0.0, 0

    for batch_idx, (data, _) in enumerate(train_loader):
      data = data.to(device)

      indices = extract_codebook_indices(quantizer, data)
      b, t = indices.shape                      # (batch_size, seq_len)

      bos_tokens = torch.full((b, 1), bos_token_id, dtype=torch.long, device=device)
      full_sequence = torch.cat([bos_tokens, indices], dim=1) # (batch_size, seq_len)

      inputs = full_sequence[:, :-1]            # [BOS, c1, c2, ..., c_{t-1}]
      targets = full_sequence[:, 1:]            # [c1, c2, ..., c_{t-1}, c_{t}]

      num_tokens = targets.numel()
      total_tokens += num_tokens

      logits = transformer(inputs)              # (batch_size, seq_len, num_embeddings)

      predictions = torch.argmax(logits, dim=-1)
      accuracy = (predictions == targets).sum().item()

      optimizer.zero_grad()

      targets = targets.reshape(-1)                             # (batch_size * seq_len)
      logits = logits.reshape(-1, transformer.num_embeddings) # (batch_size * seq_len, num_embeddings)

      loss = F.cross_entropy(logits, targets)

      loss.backward()
      optimizer.step()

      total_loss += loss.item() * num_tokens
      total_accuracy += accuracy

    train_loss = total_loss / total_tokens
    train_accuracy = (total_accuracy / total_tokens) * 100
    train_perplexity = np.exp(train_loss)

    print(f"====> Epoch {epoch} Finished")
    print(f"====> Train Loss: {train_loss:.4f} | Train Accuracy: {train_accuracy:.4f} | Train Perplexity: {train_perplexity:.2f}")

    val_loss, val_accuracy = validate(val_loader, transformer, quantizer, device)
    val_perplexity = np.exp(val_loss)
    print(f"====> Valid Loss: {val_loss:.4f} | Valid Accuracy: {val_accuracy:.4f} | Valid Perplexity: {val_perplexity:.2f}")

    if val_loss < best_val_loss:
      best_val_loss = val_loss
      torch.save({
          'epoch': epoch,
          'model_state_dict':transformer.state_dict(),
          'optimizer_state_dict': optimizer.state_dict(),
          'best_val_loss': best_val_loss,
          'perplexity': val_perplexity,
          'accuracy': val_accuracy
          }, best_save_path)
      print(f"New best model at epoch {epoch}\n")

In [ ]:
train_and_validation(train_loader, val_loader, transformer, generator, optimizer, device, epochs=50)

## Results

### Split 1

**VQ-VAE + Transformer**

Train Loss: 2.3384 | Train Accuracy: 33.2548 | Train Perplexity: 10.37

Valid Loss: 2.3886 | Valid Accuracy: 32.1847 | Valid Perplexity: 10.90

**VQGAN + Transformer**

Train Loss: 2.6044 | Train Accuracy: 29.8857 | Train Perplexity: 13.52

Valid Loss: 2.6885 | Valid Accuracy: 27.9816 | Valid Perplexity: 14.71

**VQGAN (Taming Transformers) + Transformer**

Train Loss: 1.6655 | Train Accuracy: 51.0086 | Train Perplexity: 5.29

Valid Loss: 1.6811 | Valid Accuracy: 50.8756 | Valid Perplexity: 5.37

### Split 2

**VQ-VAE + Transformer**

Train Loss: 1.7712 | Train Accuracy: 45.2070 | Train Perplexity: 5.88

Valid Loss: 1.8206 | Valid Accuracy: 43.7583 | Valid Perplexity: 6.18

**VQGAN + Transformer**

Train Loss: 3.0438 | Train Accuracy: 23.1321 | Train Perplexity: 20.98

Valid Loss: 3.2392 | Valid Accuracy: 19.5217 | Valid Perplexity: 25.51

**VQGAN (Taming Transformers) + Transformer**

Train Loss: 1.4751 | Train Accuracy: 56.6718 | Train Perplexity: 4.37

Valid Loss: 1.5256 | Valid Accuracy: 55.5467 | Valid Perplexity: 4.60

### Split 1 and 2

**VQ-VAE + Transformer**

Train Loss: 2.2196 | Train Accuracy: 35.0006 | Train Perplexity: 9.20

Valid Loss: 2.2711 | Valid Accuracy: 33.6416 | Valid Perplexity: 9.69

**VQGAN + Transformer**

Train Loss: 3.1216 | Train Accuracy: 22.0353 | Train Perplexity: 22.68

Valid Loss: 3.2254 | Valid Accuracy: 20.2094 | Valid Perplexity: 25.16

**VQGAN (Taming Transformers) + Transformer**

Train Loss: 1.5948 | Train Accuracy: 52.7040 | Train Perplexity: 4.93

Valid Loss: 1.6224 | Valid Accuracy: 52.1495 | Valid Perplexity: 5.07

# Generation

In [ ]:
checkpoint = torch.load(best_save_path, weights_only=False)
transformer.load_state_dict(checkpoint['model_state_dict'])

<All keys matched successfully>

In [ ]:
def generate_textures(transformer, quantizer, device, out_dir, num_images, batch_size=16, seq_len=1024, temperature=0.8, top_k=None):
  transformer.eval()
  quantizer.eval()

  os.makedirs(out_dir, exist_ok=True)

  h_lat = w_lat = int(math.sqrt(seq_len))
  embeddings = quantizer.vq.embeddings.weight
  embed_dim = quantizer.vq.embedding_dim
  bos_token_id = transformer.num_embeddings

  num_gen_imgs = 0
  with torch.no_grad():

    while num_gen_imgs < num_images:
      curr_batch_size = min(batch_size, num_images - num_gen_imgs)
      sequence = torch.full((curr_batch_size, 1), bos_token_id, dtype=torch.long, device=device)

      for _ in range(seq_len):
        logits = transformer(sequence)
        next_token_logits = logits[:, -1, :]                          # (batch_size, num_embeddings)
        next_token_logits /= temperature

        if top_k is not None:
          top_k = min(top_k, next_token_logits.shape[1])
          highest_scores, _ = torch.topk(next_token_logits, top_k, dim=-1)
          threshold = highest_scores[:, -1:]
          next_token_logits[next_token_logits < threshold] = -float('Inf')

        probs = F.softmax(next_token_logits, dim=-1)
        next_token = torch.multinomial(probs, 1)
        sequence = torch.cat([sequence, next_token], dim=1)

      final_indices = sequence[:, 1:]
      z = embeddings[final_indices]
      z = z.view(curr_batch_size, h_lat, w_lat, embed_dim).permute(0, 3, 1, 2).contiguous()

      generated_img = quantizer.decoder(z)
      generated_img = ((generated_img + 1.0) / 2.0).clamp(0, 1) # [-1, 1] -> [0, 1]
      generated_img = generated_img.permute(0, 2, 3, 1).cpu()

      for i in range(curr_batch_size):
        img = (generated_img[i].numpy() * 255).astype("uint8")
        image = Image.fromarray(img)
        image.save(os.path.join(out_dir, f"{num_gen_imgs + i:06d}.png"))

      num_gen_imgs += curr_batch_size
      print(f"Generated {num_gen_imgs/num_images}")

In [ ]:
temperatures = [0.8, 1.2]
top_ks = [50, 100]

for temperature in temperatures:
  for top_k in top_ks:
    print(f"Temperature: {temperature}  |   Top: {top_k}")
    output_dir = os.path.join(out_dir, f"temperature_{temperature}", f"top_{top_k}")
    generate_textures(transformer, generator, device, out_dir=output_dir, num_images=128, batch_size=16, seq_len=seq_len, temperature=temperature, top_k=top_k)
    fake_dir = os.path.join(out_dir, f"temperature_{temperature}", f"top_{top_k}")
    metrics = calculate_metrics(input1=real_dir, input2=fake_dir, cuda=torch.cuda.is_available(), fid=True, isc=False, kid=False, verbose=False)
    fid = metrics['frechet_inception_distance']
    print(f"FID: {fid:0.2f}   |   Temperature: {temperature}   |   Top-{top_k}\n")

## Results

### Split 1

**VQ-VAE + Transformer**

FID: 294.13   |   Temperature: 0.8   |   Top-50

FID: 298.97   |   Temperature: 0.8   |   Top-100

FID: 267.76   |   Temperature: 1.2   |   Top-50

FID: 273.21   |   Temperature: 1.2   |   Top-100

**VQGAN + Transformer**

FID: 252.86   |   Temperature: 0.8   |   Top-50

FID: 248.26   |   Temperature: 0.8   |   Top-100

FID: 297.67   |   Temperature: 1.2   |   Top-50

FID: 304.44   |   Temperature: 1.2   |   Top-100

**VQGAN (Taming Transformers) + Transformer**

FID: 366.49   |   Temperature: 0.8   |   Top-50

FID: 359.68   |   Temperature: 0.8   |   Top-100

FID: 363.74   |   Temperature: 1.2   |   Top-50

FID: 372.14   |   Temperature: 1.2   |   Top-100

### Split 2

**VQ-VAE + Transformer**

FID: 302.32   |   Temperature: 0.8   |   Top-50

FID: 295.50   |   Temperature: 0.8   |   Top-100

FID: 363.83   |   Temperature: 1.2   |   Top-50

FID: 363.59   |   Temperature: 1.2   |   Top-100

**VQGAN + Transformer**

FID: 245.18   |   Temperature: 0.8   |   Top-50

FID: 234.94   |   Temperature: 0.8   |   Top-100

FID: 256.70   |   Temperature: 1.2   |   Top-50

FID: 309.21   |   Temperature: 1.2   |   Top-100

**VQGAN (Taming Transformers) + Transformer**

FID: 452.33   |   Temperature: 0.8   |   Top-50

FID: 445.51   |   Temperature: 0.8   |   Top-100

FID: 426.99   |   Temperature: 1.2   |   Top-50

FID: 411.25   |   Temperature: 1.2   |   Top-100

### Split 1 and 2

**VQ-VAE + Transformer**

FID: 294.93   |   Temperature: 0.8   |   Top-50

FID: 289.32   |   Temperature: 0.8   |   Top-100

FID: 391.64   |   Temperature: 1.2   |   Top-50

FID: 386.93   |   Temperature: 1.2   |   Top-100

**VQGAN + Transformer**

FID: 257.01   |   Temperature: 0.8   |   Top-50

FID: 244.05   |   Temperature: 0.8   |   Top-100

FID: 266.53   |   Temperature: 1.2   |   Top-50

FID: 275.83   |   Temperature: 1.2   |   Top-100

**VQGAN (Taming Transformers) + Transformer**

FID: 449.10   |   Temperature: 0.8   |   Top-50

FID: 450.34   |   Temperature: 0.8   |   Top-100

FID: 409.64   |   Temperature: 1.2   |   Top-50

FID: 414.18   |   Temperature: 1.2   |   Top-100

# Reconstruction

In [ ]:
checkpoint = torch.load(best_save_path, weights_only=False)
transformer.load_state_dict(checkpoint['model_state_dict'])

<All keys matched successfully>

In [ ]:
@torch.no_grad()
def visualize_teacher_forced_reconstruction(val_loader, transformer, quantizer, device):
  for batch_idx, (data, _) in enumerate(val_loader):

    transformer.eval()
    quantizer.eval()

    transformer.to(device)
    quantizer.to(device)

    data = data.to(device)

    indices = extract_codebook_indices(quantizer, data)
    b, t = indices.shape

    bos_token_id = transformer.num_embeddings
    bos_tokens = torch.full((b, 1), bos_token_id, dtype=torch.long, device=device)
    full_sequence = torch.cat([bos_tokens, indices], dim=1)

    inputs = full_sequence[:, :-1]
    targets = full_sequence[:, 1:]

    logits = transformer(inputs)

    predictions = torch.argmax(logits, dim=-1)

    targets = targets.reshape(-1)
    logits = logits.reshape(-1, transformer.num_embeddings)

    loss = F.cross_entropy(logits, targets, reduction="sum")

    h_lat = w_lat = int(math.sqrt(seq_len))
    embeddings = quantizer.vq.embeddings.weight
    embed_dim = quantizer.vq.embedding_dim

    # quantizer reconstruction
    z_true = embeddings[indices]
    z_true = z_true.view(b, h_lat, w_lat, embed_dim).permute(0, 3, 1, 2).contiguous()
    quant_reconstructed = quantizer.decoder(z_true)
    quant_reconstructed = (quant_reconstructed + 1.0) / 2.0 # [-1, 1] -> [0, 1]
    quant_reconstructed = quant_reconstructed.permute(0, 2, 3, 1).cpu()

    # transformer reconstruction
    z_pred = embeddings[predictions]
    z_pred = z_pred.view(b, h_lat, w_lat, embed_dim).permute(0, 3, 1, 2).contiguous()
    transf_reconstructed = quantizer.decoder(z_pred)
    transf_reconstructed = (transf_reconstructed + 1.0) / 2.0 # [-1, 1] -> [0, 1]
    transf_reconstructed = transf_reconstructed.permute(0, 2, 3, 1).cpu()

    original = data.cpu()
    original = (original + 1.0) / 2.0
    original = original.permute(0, 2, 3, 1)

    if batch_idx < 5:
      fig, axes = plt.subplots(3, b, figsize=(batch_size * 4, 12))

      for i in range(b):
        axes[0, i].imshow(original[i].clamp(0, 1))
        axes[0, i].set_title(f"Original {i+1}")
        axes[0, i].axis("off")

        axes[1, i].imshow(quant_reconstructed[i].clamp(0, 1))
        axes[1, i].set_title(f"Quantizer recon. {i+1}")
        axes[1, i].axis("off")

        axes[2, i].imshow(transf_reconstructed[i].clamp(0, 1))
        axes[2, i].set_title(f"Transformer recon. {i+1}")
        axes[2, i].axis("off")
      plt.tight_layout()
      recon_save_path = os.path.join(recon_save_dir, f"batch_{batch_idx}.png")
      plt.savefig(recon_save_path, bbox_inches='tight', dpi=150)
      plt.close(fig)

    else: break

In [ ]:
visualize_teacher_forced_reconstruction(val_loader, transformer, generator, device)